In [1]:
%%capture
!pip install textwrap
!pip install transformers

##**Google - flan-T5**

Its an instruction-finetuned large language model that offers superior performance on various natural language processing tasks compared to the original T5 model.

It is a scaled-up version of Flan-T5, built on the T5 architecture, and achieves strong zero-shot and few-shot performance across a wide range of tasks, including reasoning and prompt completion.


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import textwrap

##Load the model, tockenizer, and pipeline from hugginface

In [3]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [4]:
story_generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

Device set to use cpu


In [16]:
def generate_short_story(prompt: str | None,
                         max_new_tokens: int = 220,
                         num_return_sequences: int = 1,
                         creative: bool = True):
    """
    Generate a short story from a user topic using a Flan-T5 model.

    Args:
        prompt: user topic/title or full premise
        max_new_tokens: maximum tokens to generate (NOT characters)
        num_return_sequences: number of stories to sample/return
        creative: if True, use sampling; else use beam search (more focused)
    """

    if not prompt:
        prompt = input("-Enter a topic for the story (Flan-T5): ")

    print(f"-Users prompt:\n{prompt}")

    # Keep instructions short & explicit for Flan-T5
    # Add a soft constraint to avoid generic openings and repeated lines
    final_prompt = (
        "Write a short, self-contained story about the topic below.\n"
        "Requirements:\n"
        "- 150–300 words.\n"
        "- Clear beginning, middle, and end.\n"
        "- Vivid description and emotion.\n"
        "- Natural, engaging voice.\n"
        "- Do NOT repeat sentences.\n"
        "- Do NOT start with: 'This is a story about'.\n\n"
        f"Topic: {prompt}\n\n"
        "Story:"
    )

    print(f"-Final prompt (will be given to model):\n{final_prompt}\n")
    print("(Flan-T5) Generating short story...")

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "no_repeat_ngram_size": 4,
        "repetition_penalty": 1.15,
        "num_return_sequences": num_return_sequences,
    }

    if creative:
        gen_kwargs.update({
            "do_sample": True,
            "temperature": 0.8,
            "top_p": 0.9,
            "top_k": 50,
        })
    else:
        gen_kwargs.update({
            "do_sample": False,
            "num_beams": 4,
            "length_penalty": 1.0,
            "early_stopping": True,
        })

    outputs = story_generator(final_prompt, **gen_kwargs)

    text = outputs[0]["generated_text"].strip()
    print("(Flan-T5) Generated Story:\n")
    print("\n".join(textwrap.wrap(text, width=80)))

In [17]:
generate_short_story(
    prompt = "From Tehran to Toronto, I can't make you out of my head. (from Abnzandi to Saba ...)"
)

-Users prompt:
From Tehran to Toronto, I can't make you out of my head. (from Abnzandi to Saba ...)
-Final prompt (will be given to model):
Write a short, self-contained story about the topic below.
Requirements:
- 150–300 words.
- Clear beginning, middle, and end.
- Vivid description and emotion.
- Natural, engaging voice.
- Do NOT repeat sentences.
- Do NOT start with: 'This is a story about'.

Topic: From Tehran to Toronto, I can't make you out of my head. (from Abnzandi to Saba ...)

Story:

(Flan-T5) Generating short story...
(Flan-T5) Generated Story:

When I was a kid, I lived in Tehran, Iran. I grew up in Toronto, Canada. My
parents were from Tehran, Iran, and my father was from Abnzandi, Iran. When I
was 12, I moved to Canada. I lived in Toronto for a while, and then moved back
to Iran. I've lived in Tehran for a few years, and now I'm living in Toronto. I
love living in Canada, because it's a great place to live. I can't make you out
of my head, from Tehran to Toronto.


In [19]:
generate_short_story(
    prompt = "time travel",
    max_new_tokens = 350
)

-Users prompt:
time travel
-Final prompt (will be given to model):
Write a short, self-contained story about the topic below.
Requirements:
- 150–300 words.
- Clear beginning, middle, and end.
- Vivid description and emotion.
- Natural, engaging voice.
- Do NOT repeat sentences.
- Do NOT start with: 'This is a story about'.

Topic: time travel

Story:

(Flan-T5) Generating short story...
(Flan-T5) Generated Story:

When I was a child, I dreamed of being able to travel back in time to my
childhood. I wanted to be able to do just that. I had always dreamed that I
could travel back to my childhood and live in the present. I grew up in a time
where time travel was a thing. I was able to go back in time and travel back to
that time.


In [21]:
generate_short_story(
    prompt = "Saba! I can't make you out of my head.",
    max_new_tokens = 350
)

-Users prompt:
Saba! I can't make you out of my head.
-Final prompt (will be given to model):
Write a short, self-contained story about the topic below.
Requirements:
- 150–300 words.
- Clear beginning, middle, and end.
- Vivid description and emotion.
- Natural, engaging voice.
- Do NOT repeat sentences.
- Do NOT start with: 'This is a story about'.

Topic: Saba! I can't make you out of my head.

Story:

(Flan-T5) Generating short story...
(Flan-T5) Generated Story:

Saba, I can't make you out of my head.


In [22]:
generate_short_story(
    prompt = "From Prague to Warsaw, Down with Dictators! Viva freedom!",
    max_new_tokens = 350
)

-Users prompt:
From Prague to Warsaw, Down with Dictators! Viva freedom!
-Final prompt (will be given to model):
Write a short, self-contained story about the topic below.
Requirements:
- 150–300 words.
- Clear beginning, middle, and end.
- Vivid description and emotion.
- Natural, engaging voice.
- Do NOT repeat sentences.
- Do NOT start with: 'This is a story about'.

Topic: From Prague to Warsaw, Down with Dictators! Viva freedom!

Story:

(Flan-T5) Generating short story...
(Flan-T5) Generated Story:

I was born and raised in Prague, Czechoslovakia. I grew up in a Communist
country. I was born in a communist country. I moved to the United States when I
was fourteen. I lived in the United States until I was eighteen. I was educated
at the University of California, Berkeley. I graduated from the University of
Texas at Austin with a degree in political science. I was a member of the
Student Nonviolent Coordinating Committee (SNCC). I worked as a janitor at a
local high school. I was i